<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/weeks/week-05/NB05_fairness_metrics.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 05: Algorithmic Fairness I: Harms, Metrics and Impossibility

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-05/lab.html>.

## Overview

Machine learning systems allocate loans, jobs, medical attention and police attention. This week studies how unfair outcomes arise across the machine learning life cycle, how group and individual fairness are measured, and why several natural fairness criteria cannot hold together [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to trace a fairness harm to its source in the life cycle, to compute demographic parity, equal opportunity, equalized odds and predictive parity from a confusion matrix, to state the impossibility results of Kleinberg and colleagues and of Chouldechova, and to justify the choice of a fairness criterion for a concrete application.

## Where unfairness enters

Suresh and Guttag organised sources of harm along the machine learning life cycle [1]. Historical bias exists in the world before any data is collected. Representation bias arises when some groups are under-sampled. Measurement bias arises when features or labels are poor proxies for the construct of interest. Aggregation, learning, evaluation and deployment biases arise later, when one model is fitted to heterogeneous groups, when the objective amplifies disparities, when benchmarks do not represent users and when a system is used differently from how it was designed. Mehrabi and colleagues surveyed the resulting landscape of definitions and methods [4].

Three cases show the variety of mechanisms. An investigation by ProPublica reported that a recidivism risk tool produced higher false positive rates for Black defendants than for white defendants [5]. Obermeyer and colleagues found that a widely used health algorithm predicted health care costs as a proxy for health needs, and because less money had been spent on Black patients with the same needs, the algorithm assigned them lower risk scores [6]. Buolamwini and Gebru found that commercial gender classification systems had error rates of up to 34.7 percent for darker-skinned women, while the maximum error rate for lighter-skinned men was 0.8 percent [7]. The first case concerns error rates, the second the choice of label and the third representation in training and evaluation data.

## Group fairness criteria

Most criteria compare statistics of a classifier across groups defined by a protected attribute. Demographic parity requires equal selection rates. Equalized odds requires equal true positive rates and equal false positive rates, and equal opportunity relaxes it to equal true positive rates only [8]. Predictive parity requires equal precision, so that a positive prediction means the same probability of a positive outcome in every group. Calibration within groups strengthens this to every score level.

Each criterion encodes a different view of what equal treatment means. Demographic parity treats unequal selection rates as the harm, which suits settings where historical labels themselves reflect discrimination. Equal opportunity treats unequal chances for qualified individuals as the harm. Predictive parity protects the meaning of a score for the people who use it. Dwork and colleagues proposed a complementary individual notion: Similar individuals should receive similar outcomes, where similarity is defined by a task-specific metric [9]. The difficulty moves to the metric, which must itself be justified.

## Impossibility results

Kleinberg, Mullainathan and Raghavan proved that three conditions, calibration within groups and balance of scores for the positive and for the negative class, cannot hold together except in two degenerate situations: perfect prediction or equal base rates across groups [2]. Chouldechova showed a closely related result for binary predictions: When prevalence differs between groups, a predictor with equal positive predictive value cannot also have equal false positive and false negative rates [3]. The recidivism debate illustrates both results. The tool was reported to be roughly calibrated across groups, while its error rates differed, and both observations can be true at once [3, 5].

The results do not show that fairness is hopeless. They show that fairness criteria express value judgments that must be chosen, justified and documented. Figure 5.1 shows the mechanism in its simplest form: A single threshold applied to two groups with different score distributions yields different error rates and selection rates.

![Figure 5.1](https://raw.githubusercontent.com/utkukose/machine-ethics-ai-safety-NB-lecture/main/weeks/week-05/figures/w05_fig1.png)

*Figure 5.1. Synthetic scores for two groups with different base rates. The same threshold of 0.5 yields different true positive rates, false positive rates and selection rates.*

## Fairness as a socio-technical question

Metrics are computed on data, and data carry the history of the institutions that collected them. The health algorithm studied by Obermeyer and colleagues was well calibrated for cost and still unfair for need, because cost was the wrong label [6]. A fairness audit therefore begins with questions that no metric can answer: What is the decision for, who is affected, what does each type of error cost each group and who had a voice in answering these questions [1]?

> **Pause and reflect.** In a university admission model, is a false negative (rejecting a student who would have succeeded) worse than a false positive? Does the answer differ for students from under-represented schools?

# Hands-on lab

The notebook builds a synthetic lending data set in which group B carries a historical disadvantage, trains a model that never sees the group attribute, and measures every fairness criterion of the lecture. An optional section repeats the audit on the Adult census data set when an internet connection is available [2, 8].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. A synthetic lending data set

The label is whether a loan would be repaid. Group B has lower recorded income and shorter recorded credit histories, a stylised form of historical and measurement bias. The model does not receive the group attribute.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(SEED)
n = 6000
group = rng.integers(0, 2, n)                      # 0 = group A, 1 = group B
income = rng.normal(50 - 8 * group, 12, n)
debt = rng.normal(20, 8, n)
history = rng.normal(0.6 - 0.1 * group, 0.2, n)
logit = 0.08 * (income - 45) - 0.09 * (debt - 20) + 3.0 * (history - 0.55)
repaid = (rng.uniform(size=n) < 1 / (1 + np.exp(-logit))).astype(int)
X = np.column_stack([income, debt, history])
X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(X, repaid, group, test_size=0.4, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
score = clf.predict_proba(X_te)[:, 1]
print("base rates: A =", y_te[g_te == 0].mean().round(3), " B =", y_te[g_te == 1].mean().round(3))

In [ ]:
def group_metrics(y, pred, g):
    rows = {}
    for k, name in [(0, "A"), (1, "B")]:
        m = g == k
        yk, pk = y[m], pred[m]
        tp = np.sum((pk == 1) & (yk == 1)); fp = np.sum((pk == 1) & (yk == 0))
        fn = np.sum((pk == 0) & (yk == 1)); tn = np.sum((pk == 0) & (yk == 0))
        rows[name] = {"selection rate": pk.mean(), "TPR": tp / (tp + fn), "FPR": fp / (fp + tn),
                      "PPV": tp / max(1, tp + fp), "accuracy": (tp + tn) / len(yk)}
    return pd.DataFrame(rows).T.round(3)

pred = (score >= 0.5).astype(int)
group_metrics(y_te, pred, g_te)

### Your turn, exercise 1

Compute the equal opportunity difference, TPR of group A minus TPR of group B, at threshold 0.5 without using `group_metrics`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _eo_reference(t=0.5):
    p = score >= t
    return p[(g_te == 0) & (y_te == 1)].mean() - p[(g_te == 1) & (y_te == 1)].mean()

In [ ]:
eo_difference = None  # your computation
check("Exercise 1", eo_difference, _eo_reference())

## 2. Sweeping the threshold

In [ ]:
ts = np.linspace(0.2, 0.8, 61)
dp, eo, acc = [], [], []
for t in ts:
    p = (score >= t).astype(int)
    m = group_metrics(y_te, p, g_te)
    dp.append(m.loc["A", "selection rate"] - m.loc["B", "selection rate"])
    eo.append(m.loc["A", "TPR"] - m.loc["B", "TPR"])
    acc.append(np.mean(p == y_te))
plt.figure(figsize=(7, 3.6))
plt.plot(ts, dp, label="demographic parity gap"); plt.plot(ts, eo, label="equal opportunity gap"); plt.plot(ts, acc, label="accuracy")
plt.axhline(0, color="k", lw=0.7); plt.xlabel("single threshold"); plt.legend(); plt.title("No single threshold closes both gaps"); plt.show()

## 3. Calibration within groups

The model is roughly calibrated for both groups, yet its error rates differ. This is the pattern that the impossibility results predict when base rates differ.

In [ ]:
bins = np.linspace(0, 1, 11)
for k, name in [(0, "A"), (1, "B")]:
    m = g_te == k
    idx = np.digitize(score[m], bins) - 1
    xs = [score[m][idx == b].mean() for b in range(10) if np.sum(idx == b) > 20]
    ys = [y_te[m][idx == b].mean() for b in range(10) if np.sum(idx == b) > 20]
    plt.plot(xs, ys, marker="o", label=f"group {name}")
plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.xlabel("mean score in bin"); plt.ylabel("observed repayment rate")
plt.legend(); plt.title("Calibration by group"); plt.show()

### Your turn, exercise 2

Keep threshold 0.5 for group A and search for a threshold for group B that makes the absolute equal opportunity gap smaller than 0.02. Report the accuracy before and after.

In [ ]:
t_B = 0.5  # change this value
p = np.where(g_te == 0, score >= 0.5, score >= t_B).astype(int)
gap = p[(g_te == 0) & (y_te == 1)].mean() - p[(g_te == 1) & (y_te == 1)].mean()
print("gap:", round(gap, 4), " accuracy:", round(np.mean(p == y_te), 4))
check("Exercise 2", abs(gap) < 0.02, True)

## 4. Optional: the Adult census data set

This section needs an internet connection and is skipped otherwise.

In [ ]:
try:
    from sklearn.datasets import fetch_openml
    adult = fetch_openml("adult", version=2, as_frame=True)
    df = adult.frame.dropna()
    Xa = pd.get_dummies(df.drop(columns=["class", "sex", "race"]), drop_first=True).astype(float)
    ya = (df["class"] == ">50K").astype(int).to_numpy()
    ga = (df["sex"] == "Female").astype(int).to_numpy()
    Xa_tr, Xa_te, ya_tr, ya_te, ga_tr, ga_te = train_test_split(Xa, ya, ga, test_size=0.3, random_state=0)
    m = LogisticRegression(max_iter=2000).fit((Xa_tr - Xa_tr.mean()) / (Xa_tr.std() + 1e-9), ya_tr)
    pa = m.predict((Xa_te - Xa_tr.mean()) / (Xa_tr.std() + 1e-9))
    display(group_metrics(ya_te, pa, ga_te).rename(index={"A": "male", "B": "female"}))
except Exception as e:
    print("Optional section skipped:", type(e).__name__)

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-05/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which source of harm in the framework of Suresh and Guttag is most likely in a data set from your field, and how would you detect it?
2. Choose one fairness criterion for a decision system you know and defend it against the other criteria in three sentences.
3. The impossibility results turn fairness into a choice. Who should make that choice for a public-sector system, and how should it be documented?

## Weekly task and submission

Write a fairness audit of about 500 words for the synthetic lending model or for a model and data set from your field. Report at least four group metrics, discuss which criterion matters most for the decision and why, and state one limitation of the audit. Attach the executed notebook.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Fairness definitions in a regulated domain.** Choose credit, hiring or health care and review which fairness definitions regulators, auditors and practitioners actually use. Relate the choices to the impossibility results of this week [2, 3].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Suresh, H., & Guttag, J. (2021). A framework for understanding sources of harm throughout the machine learning life cycle. In *Equity and Access in Algorithms, Mechanisms, and Optimization (EAAMO '21)*. ACM. <https://doi.org/10.1145/3465416.3483305>

[2] Kleinberg, J., Mullainathan, S., & Raghavan, M. (2017). Inherent trade-offs in the fair determination of risk scores. In *8th Innovations in Theoretical Computer Science Conference (ITCS 2017), LIPIcs 67* (pp. 43:1-43:23). <https://doi.org/10.4230/LIPIcs.ITCS.2017.43>

[3] Chouldechova, A. (2017). Fair prediction with disparate impact: A study of bias in recidivism prediction instruments. *Big Data*, 5(2), 153-163. <https://doi.org/10.1089/big.2016.0047>

[4] Mehrabi, N., Morstatter, F., Saxena, N., Lerman, K., & Galstyan, A. (2021). A survey on bias and fairness in machine learning. *ACM Computing Surveys*, 54(6), Article 115. <https://doi.org/10.1145/3457607>

[5] Angwin, J., Larson, J., Mattu, S., & Kirchner, L. (2016). *Machine bias*. ProPublica. <https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing>

[6] Obermeyer, Z., Powers, B., Vogeli, C., & Mullainathan, S. (2019). Dissecting racial bias in an algorithm used to manage the health of populations. *Science*, 366(6464), 447-453. <https://doi.org/10.1126/science.aax2342>

[7] Buolamwini, J., & Gebru, T. (2018). Gender shades: Intersectional accuracy disparities in commercial gender classification. In *Proceedings of the 1st Conference on Fairness, Accountability and Transparency, PMLR 81* (pp. 77-91).

[8] Hardt, M., Price, E., & Srebro, N. (2016). Equality of opportunity in supervised learning. In *Advances in Neural Information Processing Systems 29*.

[9] Dwork, C., Hardt, M., Pitassi, T., Reingold, O., & Zemel, R. (2012). Fairness through awareness. In *Proceedings of the 3rd Innovations in Theoretical Computer Science Conference* (pp. 214-226). ACM. <https://doi.org/10.1145/2090236.2090255>